# Parquet particionado por zona con PySpark

Reutiliza el Parquet de pedidos que guardaste en el [cuaderno 05](05_csv_python_vs_pyspark.ipynb). Verás cómo Spark organiza el resultado en carpetas por zona y cómo un filtro permite leer solo las zonas solicitadas. Ejecuta ambos cuadernos en el mismo entorno para que la ruta siga disponible.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("Parquet por zona").getOrCreate()

ruta_parquet = Path(input("Pega la ruta de Parquet del cuaderno 05: ").strip()).resolve()
pedidos = spark.read.parquet(str(ruta_parquet))

## 1. Guardar los pedidos por zona

Con `partitionBy("zona")`, Spark agrupa los archivos en carpetas como `zona=zona_3`. La salida se guarda junto al Parquet del cuaderno 05 y se reemplaza al repetir esta celda.

In [ ]:
ruta_particionada = ruta_parquet.with_name("pedidos_parquet_por_zona")
pedidos.write.mode("overwrite").partitionBy("zona").parquet(str(ruta_particionada))

print(sorted(carpeta.name for carpeta in ruta_particionada.glob("zona=*")))

## 2. Consultar una zona

Lee desde la carpeta raíz para que Spark recupere la columna `zona` a partir de los nombres de las carpetas. En el plan de ejecución, busca `PartitionFilters` para ver el filtro aplicado a esas carpetas.

In [ ]:
pedidos_por_zona = spark.read.parquet(str(ruta_particionada))
pedidos_zona_7 = pedidos_por_zona.filter(F.col("zona") == "zona_7")

pedidos_zona_7.show(5)
pedidos_zona_7.explain()

Prueba con otra zona y comprueba cómo cambia el filtro en el plan de ejecución.

In [ ]:
spark.stop()

## Navegación

[Anterior · CSV vs. Parquet](05_csv_python_vs_pyspark.ipynb) · [Volver al README](../README.md) · [Siguiente · Spark y HDFS](07_spark_hdfs.ipynb)